<a href="https://colab.research.google.com/github/427paul/AI_Agent/blob/main/%5BBDA%5D_14%EC%A3%BC%EC%B0%A8_LangGraph_%EA%B8%B0%EB%B3%B8_%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ✏️ 종합 과제

배운 걸 합쳐, **숫자 맞히기 판정 그래프**를 만들어 보세요.

- State: `{"number": int, "log": Annotated[list, operator.add]}`
- 노드 `judge`: number가 100보다 크면 log에 "너무 큼", 작으면 "너무 작음", 같으면 "정답!" 추가
- (심화) 조건 분기로 "정답이면 END, 아니면 다시 판정"하는 루프 구조로 확장

> 💡 막히면 Part 3(Reducer) + Part 4(분기) + Part 5(루프)를 조합하세요.

In [1]:
!pip install -q langgraph langchain-google-genai langchain-core

print("✅ 설치 완료!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.8/72.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 11.5 MB/s eta 0:00:00
✅ 설치 완료!


In [2]:
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass("Google API 키를 입력하세요: ")
print("✅ API 키 설정 완료!")

Google API 키를 입력하세요: ··········
✅ API 키 설정 완료!


In [15]:
import operator
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

# ① State 정의 — 공유 메모장의 모양
class State(TypedDict):
    number: int
    log : Annotated[list, operator.add]

# ② Node 정의 — text를 대문자로 바꿔 돌려줌
def judge(state: State):
    num = state['number']

    if num > 100:
      message = f"[{num} 너무 큼]"
    elif num < 100:
      messgae = f"[{num} 너무 작음]"
    else:
      message = f"[{num} 정답!]"

    return {"log": [message]}

def route_check(state: State):
  last_message = state["log"][-1]
  if "정답!" in last_message:
    return 'end'
  else:
    return "retry"

def adjust_number(state: State):
  num = state['number']
  if num > 100:
    return {"number" : num - 10}
  elif num < 100:
    return {"number" : num + 10}
  else:
    return {"number" : num}

builder = StateGraph(State)
builder.add_node("judge", judge)
builder.add_node("adjust", adjust_number)

builder.add_edge(START, "judge")

builder.add_conditional_edges(
    "judge",
    route_check,
    {
        "retry" : "adjust",
        "end" : END
    }
)

builder.add_edge("adjust", "judge")

graph = builder.compile()

In [16]:
result = graph.invoke({"number": 120, "log": []})

print("최종 숫자:", result["number"])
print("\n--- 판정 및 실행 로그 ---")
for log_item in result["log"]:
    print(log_item)

최종 숫자: 100

--- 판정 및 실행 로그 ---
[120 너무 큼]
[110 너무 큼]
[100 정답!]
